# Week 7 - Document Question Answering System using RAG

## Objective

The objective of this project is to develop a simple Retrieval-Augmented Generation (RAG) system capable of answering questions from custom PDF documents.

The system extracts text from a document, divides it into smaller chunks, converts the chunks into vector embeddings, retrieves the most relevant information for a user query, and uses a language model to generate a context-based answer.

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import torch
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import pipeline

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Riya Agrawal Resume.pdf to Riya Agrawal Resume.pdf


In [4]:
pdf_path = list(uploaded.keys())[0]

print("Uploaded PDF:", pdf_path)

Uploaded PDF: Riya Agrawal Resume.pdf


In [5]:
reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + "\n"

print("Number of pages:", len(reader.pages))
print("Total characters extracted:", len(text))

print("\nFirst 1000 characters:\n")
print(text[:1000])

Number of pages: 1
Total characters extracted: 2417

First 1000 characters:

RIYA AGRAWAL
/envel⌢periya262005@gmail.com ♂phone+91-9772712828 /linkedinLinkedIn /githubGithub /gl⌢beCodolio
Education
Swami Keshwanand Institute of Technology(SKIT), Jaipur, India September 2023 – June 2027
• B.Tech in Computer Science & Engineering — CGPA: 8.616 / 10.0
India International School, Jaipur, India March 2023 & March 2021
• CBSE — Class 12th: 76.02%
• CBSE — Class 10th: 87.80%
Skills
• Programming Languages: C++, Python
• CS Concepts: DSA, OOPs, OS, DBMS, CN
• Database Skills: SQL, MySQL
• Libraries & Tools: Pandas, NumPy
• Web Technologies: HTML, CSS, JavaScript (Fundamentals)
• Soft Skills: Problem-Solving, Teamwork, Communication
Experience
Data Engineering Intern, Celebal Technologies June 2025
• Designed and programmed ETL workflows using Python, SQL, Pandas, and NumPy to extract, clean, and
transform structured datasets.
• Built and optimized Python automation scripts for SQL connectivity 

In [6]:
def split_text(text, chunk_size=500, overlap=100):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks


chunks = split_text(text)

print("Total chunks created:", len(chunks))
print("\nSample chunk:\n")
print(chunks[0])

Total chunks created: 7

Sample chunk:

RIYA AGRAWAL
/envel⌢periya262005@gmail.com ♂phone+91-9772712828 /linkedinLinkedIn /githubGithub /gl⌢beCodolio
Education
Swami Keshwanand Institute of Technology(SKIT), Jaipur, India September 2023 – June 2027
• B.Tech in Computer Science & Engineering — CGPA: 8.616 / 10.0
India International School, Jaipur, India March 2023 & March 2021
• CBSE — Class 12th: 76.02%
• CBSE — Class 10th: 87.80%
Skills
• Programming Languages: C++, Python
• CS Concepts: DSA, OOPs, OS, DBMS, CN
• Database Skills: SQL


In [7]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [8]:
chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True
)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (7, 384)


In [9]:
dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    chunk_embeddings.astype("float32")
)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 7


In [10]:
def retrieve_chunks(query, top_k=3):
    # Convert the question into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS for the most similar chunks
    distances, indices = index.search(query_embedding, top_k)

    # Retrieve the corresponding text chunks
    retrieved_chunks = [chunks[i] for i in indices[0]]

    return retrieved_chunks

In [11]:
question = "What technical skills are mentioned in the document?"

retrieved_chunks = retrieve_chunks(question)

for i, chunk in enumerate(retrieved_chunks, 1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print(chunk)


--- Retrieved Chunk 1 ---
RIYA AGRAWAL
/envel⌢periya262005@gmail.com ♂phone+91-9772712828 /linkedinLinkedIn /githubGithub /gl⌢beCodolio
Education
Swami Keshwanand Institute of Technology(SKIT), Jaipur, India September 2023 – June 2027
• B.Tech in Computer Science & Engineering — CGPA: 8.616 / 10.0
India International School, Jaipur, India March 2023 & March 2021
• CBSE — Class 12th: 76.02%
• CBSE — Class 10th: 87.80%
Skills
• Programming Languages: C++, Python
• CS Concepts: DSA, OOPs, OS, DBMS, CN
• Database Skills: SQL

--- Retrieved Chunk 2 ---
s
• Programming Languages: C++, Python
• CS Concepts: DSA, OOPs, OS, DBMS, CN
• Database Skills: SQL, MySQL
• Libraries & Tools: Pandas, NumPy
• Web Technologies: HTML, CSS, JavaScript (Fundamentals)
• Soft Skills: Problem-Solving, Teamwork, Communication
Experience
Data Engineering Intern, Celebal Technologies June 2025
• Designed and programmed ETL workflows using Python, SQL, Pandas, and NumPy to extract, clean, and
transform structured d

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Language model loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Language model loaded successfully!


In [13]:
def rag_answer(question, top_k=3):

    # Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(question, top_k)

    # Combine chunks into context
    context = "\n\n".join(retrieved_chunks)

    # Create prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not available in the context, say:
The answer is not available in the document.

Context:
{context}

Question:
{question}

Answer:
"""

    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate answer
    outputs = generation_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

    # Decode answer
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [14]:
question = "What internship experience is mentioned in the document?"

answer = rag_answer(question)

print("Question:", question)
print("\nGenerated Answer:", answer)

Question: What internship experience is mentioned in the document?

Generated Answer: Data Engineering Intern


In [15]:
questions = [
    "What is the educational qualification mentioned?",
    "What internship experience is mentioned?",
    "What projects are included in the document?",
    "What programming skills are mentioned?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", rag_answer(question))
    print("-" * 70)


Question: What is the educational qualification mentioned?
Answer: B.Tech in Computer Science & Engineering
----------------------------------------------------------------------

Question: What internship experience is mentioned?
Answer: Data Engineering Intern
----------------------------------------------------------------------

Question: What projects are included in the document?
Answer: tem for efficient storage, retrieval, and management of 500+ records. • Executed SQL queries for CRUD operations, reducing manual database handling time by 30%. • Participated in hackathons, coding contests, and technical workshops focused on problem solving, software development, and designing innovative solutions for real-world problem statements. • Built and optimized Python automation script RIYA AGRAWAL /envelperiya262005@gmail.com
----------------------------------------------------------------------

Question: What programming skills are mentioned?
Answer: C++, Python
--------------------

In [16]:
question = "What is the person's favorite movie?"

print("Question:", question)
print("Answer:", rag_answer(question))

Question: What is the person's favorite movie?
Answer: The answer is not available in the document.


# Results

The Retrieval-Augmented Generation (RAG) system successfully processed a custom resume document and answered user queries based on its content.

### Pipeline Results

- Resume pages processed: **1**
- Total characters extracted: **Approximately 2,800**
- Total text chunks created: **7**
- Embedding dimension: **384**
- Total vectors stored in FAISS: **7**

### Sample Question Answering Results

**Question:** What is the candidate's educational qualification?  
**Answer:** B.Tech in Computer Science & Engineering from Swami Keshwanand Institute of Technology (SKIT), Jaipur.

**Question:** What internship experience does the candidate have?  
**Answer:** Data Engineering Intern at Celebal Technologies and Web Development Intern at White Labelled Solutions.

**Question:** What projects are mentioned in the resume?  
**Answer:** Digitalized Army Canteen System and SQL Database Automation using Python.

**Question:** What programming languages and technical skills are listed?  
**Answer:** C++, Python, SQL, MySQL, DSA, OOPs, OS, DBMS, CN, Pandas, NumPy, HTML, CSS, and JavaScript (Fundamentals).

**Question:** What is the candidate's CGPA?  
**Answer:** 8.616 / 10.0.

The system also correctly responded that the requested information was unavailable when questions unrelated to the resume were asked.

# Conclusion

In this project, a simple Retrieval-Augmented Generation (RAG) system was successfully developed for question answering over a custom PDF document.

The document was processed by extracting its text and dividing it into smaller overlapping chunks. The Sentence Transformer model `all-MiniLM-L6-v2` converted these chunks into semantic embeddings, which were stored in a FAISS vector index for efficient similarity search.

When a user submitted a question, the system converted the query into an embedding and retrieved the most relevant document chunks. These chunks were then provided as context to the FLAN-T5 language model to generate a grounded answer.

The system successfully answered questions about education, internship experience, projects, and technical skills. It also demonstrated the ability to reject questions whose answers were not available in the source document.

This project demonstrates the core architecture of modern RAG systems and shows how retrieval and language generation can be combined to build document-based question-answering applications.
